## LangGraph SFT Repair Workflow

This notebook repairs broken Python code using a **step-by-step LangGraph workflow**.
It is designed to be understandable for both beginners and advanced users.

### What this notebook does

1. Starts a local GGUF model with `llama.cpp` `llama-server` (OpenAI-compatible API).
2. Runs an automated repair loop: attempt fix -> validate -> choose next strategy.
3. Uses smarter fallback strategies when needed (RAG, reflection, external expert).
4. Stops safely with clear stop rules (success, max attempts, repeated failures, or no progress).

### Why this workflow is useful

- **Local-first**: your main repair model runs locally.
- **Structured routing**: different error types use different strategies.
- **Traceable**: optional LangSmith logging for model calls.
- **Bounded**: avoids infinite repair loops.

### High-level flow

```
Initial SFT Repair (local)
        |
        v
Compile + Runtime Checks
   | pass         | fail
   v              v
Success      Diagnose Error Type
                    |
                    v
             Choose Next Strategy

Syntax/Name/Timeout:
  Reflection Critic -> SFT (with reflection hints) -> Recheck
  If same error repeats -> External Expert -> Recheck

API/Library:
  RAG (retrieve -> assess -> summarize) -> SFT -> Recheck
  If still failing -> Reflection -> SFT -> Recheck
  If still failing -> External Expert -> Recheck

Logic/Shape/Type:
  Reflection Critic -> SFT -> Recheck
  If still failing -> RAG -> SFT -> Recheck
  If same/persistent failure -> External Expert -> Recheck

Stop Conditions:
  Success | Max attempts | No meaningful progress | Strategies exhausted
```

### Strategy policy (simple view)

- **Syntax/Name/Timeout errors**: reflection first, then external expert.
- **API/Library errors**: RAG first (documentation helps most).
- **Logic/shape/type errors**: reflection first, then RAG or external expert.

### Before you run

- Required: Python environment with notebook dependencies installed.
- Required: access to `llama-server` (already on PATH or buildable from `llama.cpp`).
- Optional: `OPENROUTER_API_KEY` for remote fallback models.
- Optional: `LANGCHAIN_API_KEY` for LangSmith tracing.

### What you will get at the end

- Final repaired code.
- Final status (`success` or `failure`).
- Number of attempts.
- Route history showing which strategies were used.
- Stop reason when repair is unsuccessful.

### 0) Setup Guide: Local model and server

This section prepares the **local inference server** used by the rest of the notebook.
Later cells call this server through `ChatOpenAI(base_url=LLAMA_SERVER_URL, ...)`.

### Step A: Prepare llama-server (Windows)

Choose one option:

- **Option A (recommended)**: install a prebuilt `llama-server` binary and make sure it is available on PATH.
- **Option B**: let the notebook build from source (requires CMake + MSVC Build Tools).

### Step B: Choose model source

You can use either:

- **Local GGUF file**:
  - set `LOCAL_GGUF_PATH` to your `.gguf` file path
- **Hugging Face GGUF**:
  - simple: set `HF_GGUF_MODEL` (default is `H4miid/mistral-7b-instruct-v0.3.Q4_K_M.gguf`)
  - explicit: set `HF_GGUF_REPO_ID` + `HF_GGUF_FILENAME`

### Step C: (Optional) Private HF access

If your Hugging Face repo needs auth, set one of:

- `HF_TOKEN`
- `HUGGINGFACEHUB_API_TOKEN`

### Step D: Server output used downstream

When setup succeeds, these values are available for the rest of the notebook:

- `LLAMA_SERVER_PATH` (path to executable)
- `LLAMA_SERVER_URL` (OpenAI-compatible endpoint)

### Recommended execution order for startup

1. Step 1: Prepare llama.cpp server tools.
2. Step 2: Locate/build llama-server binary.
3. Step 3: Load dependencies and tracing setup.
4. Step 4: Apply optional SSL truststore fix.
5. Step 5: Resolve GGUF model and start llama-server.

If Step 5 reports healthy server status, continue with workflow cells below.

### 3) Load libraries and tracing setup
```
# =============================================================================
# dependency imports + environment + LangSmith tracing
# =============================================================================
# Goal: Load required packages and initialize observability/tracing once.
# Output: Ready runtime context for all later workflow cells.
```

In [31]:
# %pip install langgraph langchain langchain-openai langsmith python-dotenv huggingface_hub openai requests chromadb sentence-transformers duckduckgo-search
import warnings
warnings.filterwarnings("ignore")

import difflib
import json
import os
import re
import subprocess
import sys
import tempfile
import traceback as tb
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, TypedDict

import chromadb
import requests
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from dotenv import load_dotenv
from langgraph.graph import END, StateGraph
from sentence_transformers import CrossEncoder

# LangChain imports for LangSmith tracing support
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

# Optional: DuckDuckGo Search (install with: pip install duckduckgo-search)
try:
    from ddgs import DDGS
    DDGS_AVAILABLE = True
except ImportError:
    DDGS_AVAILABLE = False
    print("[Warning] duckduckgo-search not installed. Web search will return placeholder results.")

# =============================================================================
# LangSmith Setup - Create Project and Enable Tracing
# =============================================================================
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT_NAME = "MentorApp-LangGraph-SFT-Repair"

def setup_langsmith():
    """Configure LangSmith tracing and verify/create the project."""
    if not LANGCHAIN_API_KEY:
        os.environ["LANGCHAIN_TRACING_V2"] = "false"
        print("ℹ LangSmith tracing disabled (add LANGCHAIN_API_KEY to .env to enable)")
        return False
    
    # Set environment variables for LangSmith
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
    os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT_NAME
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    
    # Verify connection and project using LangSmith API
    try:
        from langsmith import Client
        client = Client(api_key=LANGCHAIN_API_KEY)
        
        # Check if project exists, create if not
        try:
            projects = list(client.list_projects())
            project_names = [p.name for p in projects]
            
            if LANGSMITH_PROJECT_NAME not in project_names:
                # Create new project
                client.create_project(LANGSMITH_PROJECT_NAME)
                print(f"✓ Created new LangSmith project: {LANGSMITH_PROJECT_NAME}")
            else:
                print(f"✓ LangSmith project exists: {LANGSMITH_PROJECT_NAME}")
                
        except Exception as e:
            # Project listing might fail on older API versions, but tracing still works
            print(f"ℹ LangSmith project check: {e}")
        
        print(f"✓ LangSmith tracing enabled → project: {LANGSMITH_PROJECT_NAME}")
        print(f"  Dashboard: https://smith.langchain.com/o/default/projects/{LANGSMITH_PROJECT_NAME}")
        return True
        
    except ImportError:
        print("ℹ langsmith package not installed. Install with: pip install langsmith")
        print("✓ Basic LangChain tracing enabled (project auto-created on first trace)")
        return True
    except Exception as e:
        print(f"⚠ LangSmith setup warning: {e}")
        print("✓ LangSmith tracing enabled (project will be created on first trace)")
        return True

LANGSMITH_ENABLED = setup_langsmith()

✓ LangSmith project exists: MentorApp-LangGraph-SFT-Repair
✓ LangSmith tracing enabled → project: MentorApp-LangGraph-SFT-Repair
  Dashboard: https://smith.langchain.com/o/default/projects/MentorApp-LangGraph-SFT-Repair


### 1) Prepare llama.cpp server tools


llama.cpp setup (CPU) + llama-server discovery/build

Goal: Provide an OpenAI-compatible endpoint for a local GGUF model.

Strategy: Prefer an existing `llama-server`; build only if missing.


In [5]:
# Clone llama.cpp and build it with CUDA support.
# This only needs to run once — subsequent runs skip the clone/build.

LLAMA_CPP_DIR = Path("../llama.cpp").resolve()

# Step 1: Clone the repo (skip if it already exists)
if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp …")
    subprocess.run(
        ["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)],
        check=True,
    )
    print(f"✓ Cloned → {LLAMA_CPP_DIR}")
else:
    print(f"✓ llama.cpp already at {LLAMA_CPP_DIR}")

# Step 2: CMake configure + build
build_dir = LLAMA_CPP_DIR / "build"
build_dir.mkdir(exist_ok=True)

print("Configuring with CUDA …")
subprocess.run(["cmake", "..", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"],
               cwd=str(build_dir), check=True)

print("Building (may take a few minutes) …")
subprocess.run(["cmake", "--build", ".", "--config", "Release", "-j"],
               cwd=str(build_dir), check=True)

# Step 3: Locate the server executable
candidates = [
    build_dir / "bin" / "Release" / "llama-server.exe",  # Windows MSVC
    build_dir / "bin" / "llama-server.exe",               # Windows Ninja
    build_dir / "bin" / "llama-server",                   # Linux / macOS
]
LLAMA_SERVER_EXE = next((p for p in candidates if p.exists()), None)

# Fallback: recursive search
if not LLAMA_SERVER_EXE:
    hits = [f for f in build_dir.rglob("llama-server*") if f.is_file() and f.suffix in ("", ".exe")]
    if hits:
        LLAMA_SERVER_EXE = hits[0]

if LLAMA_SERVER_EXE:
    print(f"✓ Server executable: {LLAMA_SERVER_EXE}")
else:
    raise FileNotFoundError("llama-server not found after build — check CMake output.")

Cloning llama.cpp …


Cloning into '/workspace/llama.cpp'...


✓ Cloned → /workspace/llama.cpp
Configuring with CUDA …
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.43.0") 


CMAKE_BUILD_TYPE=Release


-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5") 
-- Found OpenMP_CXX: -fopenmp (found version "4.5") 
-- Found OpenMP: TRUE (found version "4.5")  
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "13.0.88") 
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 13.0.88
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compi

### 4) Optional SSL truststore fix
```
# =============================================================================
# optional SSL truststore injection
# =============================================================================
# Goal: Improve HTTPS reliability in restrictive Windows environments.
# Behavior: Attempts truststore injection; safely continues if unavailable.
```

In [6]:
# Optional: improve SSL reliability on Windows corporate environments
try:
    import truststore

    truststore.inject_into_ssl()
    print("Injected OS certificate store (truststore) for SSL verification.")
except Exception as e:
    print(f"truststore not enabled ({type(e).__name__}: {e})")

Injected OS certificate store (truststore) for SSL verification.


### 5) Resolve GGUF model and start llama-server
```
# =============================================================================
# GGUF model resolution + llama-server startup (CPU)
# =============================================================================
# Goal: Resolve a local/HF GGUF model and launch an OpenAI-compatible server.
# Output: `LLAMA_SERVER_URL` exported for LangChain model calls.
```

In [75]:
# =============================================================================
# HF GGUF model (GPU) + start llama-server
# =============================================================================
# This section is intentionally *GPU-first*.
# You can:
# - Point at a local `.gguf` file, OR
# - Download a `.gguf` from Hugging Face (repo_id + filename)
# Then start `llama-server` which exposes an OpenAI-compatible API (used later by `ChatOpenAI`).

import os
import subprocess
import time
from pathlib import Path

import requests
from huggingface_hub import hf_hub_download

# ----------------------------
# Model source (pick one)
# ----------------------------
# Option A: Local GGUF path
GGUF_MODELS_DIR = Path("./models/gguf").resolve()
GGUF_MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_GGUF_PATH = os.getenv("LOCAL_GGUF_PATH", "").strip()

# Option B: Hugging Face model
# Supports either:
# - HF_GGUF_MODEL="owner/repo" (single-file GGUF repo), or
# - HF_GGUF_REPO_ID + HF_GGUF_FILENAME
HF_GGUF_MODEL = os.getenv("HF_GGUF_MODEL", "H4miid/qwen2_5_coder_7b_merged_f16.gguf").strip()
HF_GGUF_REPO_ID = os.getenv("HF_GGUF_REPO_ID", "").strip()
HF_GGUF_FILENAME = os.getenv("HF_GGUF_FILENAME", "").strip()
if not HF_GGUF_REPO_ID:
    HF_GGUF_REPO_ID = HF_GGUF_MODEL
if not HF_GGUF_FILENAME:
    # For single-file GGUF repos, infer filename from repo name.
    HF_GGUF_FILENAME = HF_GGUF_REPO_ID.rsplit("/", 1)[-1]

HF_GGUF_REVISION = os.getenv("HF_GGUF_REVISION", "main").strip()
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

# ----------------------------
# Server config
# ----------------------------
LLAMA_SERVER_HOST = os.getenv("LLAMA_SERVER_HOST", "127.0.0.1").strip()
LLAMA_SERVER_PORT = int(os.getenv("LLAMA_SERVER_PORT", "8081"))
# Increased default context to reduce truncation on longer prompts + code outputs.
LLAMA_SERVER_CTX = int(os.getenv("LLAMA_SERVER_CTX", "8192"))
LLAMA_SERVER_THREADS = int(os.getenv("LLAMA_SERVER_THREADS", str(max(1, (os.cpu_count() or 4) - 1))))
LLAMA_SERVER_N_GPU_LAYERS = int(os.getenv("LLAMA_SERVER_N_GPU_LAYERS", "999"))

# OpenAI-compatible base URL used by LangChain
LLAMA_SERVER_URL = f"http://{LLAMA_SERVER_HOST}:{LLAMA_SERVER_PORT}/v1"
print(f"[Config] LLAMA_SERVER_CTX={LLAMA_SERVER_CTX}")

[Config] LLAMA_SERVER_CTX=8192


In [33]:
def resolve_gguf_path() -> Path:
    """Return the GGUF path either from a local file or from HF Hub."""
    if LOCAL_GGUF_PATH:
        p = Path(LOCAL_GGUF_PATH).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"LOCAL_GGUF_PATH does not exist: {p}")
        return p

    GGUF_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    target = GGUF_MODELS_DIR / HF_GGUF_FILENAME
    if target.exists():
        print(f"Using cached GGUF: {target}")
        return target

    print(f"Downloading from HF: {HF_GGUF_REPO_ID} / {HF_GGUF_FILENAME}")
    downloaded = hf_hub_download(
        repo_id=HF_GGUF_REPO_ID,
        filename=HF_GGUF_FILENAME,
        revision=HF_GGUF_REVISION,
        token=HF_TOKEN,
        local_dir=str(GGUF_MODELS_DIR),
        local_dir_use_symlinks=False,
    )
    return Path(downloaded).resolve()

GGUF_MODEL_PATH = resolve_gguf_path()

In [35]:
def _server_health_ok(host: str, port: int) -> bool:
    try:
        r = requests.get(f"http://{host}:{port}/health", timeout=2)
        return r.status_code == 200
    except Exception:
        return False


def start_llama_server(model_path: Path) -> subprocess.Popen | None:
    """Start llama-server if it is not already running."""
    if _server_health_ok(LLAMA_SERVER_HOST, LLAMA_SERVER_PORT):
        print(f"llama-server already running: {LLAMA_SERVER_URL}")
        return None

    cmd = [
        str(LLAMA_SERVER_EXE),
        "--model",
        str(model_path),
        "--host",
        LLAMA_SERVER_HOST,
        "--port",
        str(LLAMA_SERVER_PORT),
        "--ctx-size",
        str(LLAMA_SERVER_CTX),
        "--n-gpu-layers",
        str(LLAMA_SERVER_N_GPU_LAYERS),
        "--threads",
        str(LLAMA_SERVER_THREADS),
    ]

    print("Starting llama-server...")
    print(" ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    print("Waiting for /health...")
    for _ in range(60):
        time.sleep(1)
        if _server_health_ok(LLAMA_SERVER_HOST, LLAMA_SERVER_PORT):
            print(f"llama-server is ready: {LLAMA_SERVER_URL}")
            return proc

    print("llama-server did not become healthy in time.")
    return proc


LLAMA_SERVER_PROCESS = start_llama_server(GGUF_MODEL_PATH)

# Make it available to downstream cells
os.environ["LLAMA_SERVER_URL"] = LLAMA_SERVER_URL
print(f"LLAMA_SERVER_URL = {LLAMA_SERVER_URL}")

llama-server already running: http://127.0.0.1:8081/v1
LLAMA_SERVER_URL = http://127.0.0.1:8081/v1


### 6) Define workflow state and utilities

workflow state schema + shared utility helpers

Goal: Define a single typed state object and reusable helper functions.

Benefit: Consistent routing, tracking, and stop-condition handling.


In [99]:
# =============================================================================
# State Definition & Utilities
# =============================================================================

class RepairState(TypedDict):
    """Central state object passed between all nodes in the workflow."""
    # Core code tracking
    original_code: str
    current_code: str
    traceback: str
    check_result: dict[str, Any]
    
    # Attempt tracking
    attempt_count: int
    max_attempts: int
    attempt_history: list[dict[str, Any]]
    route_history: list[str]
    
    # Error classification (for intelligent routing)
    error_category: str  # syntax_error, name_error, timeout, api_library_error, local_reasoning_error
    
    # Strategy flags (each used at most once)
    used_traceback: bool
    used_rag: bool
    used_web: bool
    used_reflection: bool
    used_external: bool
    
    # RAG & context data
    local_docs: list[str]
    web_docs: list[str]
    local_context_quality: str  # "good" or "weak"
    summarized_hints: str
    reflection_feedback: dict[str, Any]
    initial_traceback: str
    error_type: str
    error_explanation: str
    
    # Flow control
    next_strategy: str
    failure_signature: str
    previous_failure_signature: str
    repeated_failure_count: int
    no_meaningful_change_count: int
    should_stop: bool
    stop_reason: str
    
    # Final output
    final_status: str  # "success" or "failure"
    final_code: str


# --- Utility Functions ---

def now_iso() -> str:
    """Return current UTC timestamp in ISO format."""
    return datetime.now(timezone.utc).isoformat()


def normalize_code(code: str) -> str:
    """Normalize code for comparison (strip trailing whitespace per line)."""
    return "\n".join(line.rstrip() for line in code.strip().splitlines())


def code_changed_meaningfully(before: str, after: str, ratio_threshold: float = 0.995) -> bool:
    """
    Check if code changed meaningfully (not just whitespace).
    Returns False if code is identical or nearly identical (ratio > threshold).
    """
    if normalize_code(before) == normalize_code(after):
        return False
    ratio = difflib.SequenceMatcher(a=before, b=after).ratio()
    return ratio < ratio_threshold


def extract_failure_signature(traceback_text: str) -> str:
    """Extract the final error line from traceback as a 'signature'."""
    lines = [line.strip() for line in traceback_text.splitlines() if line.strip()]
    return lines[-1] if lines else "unknown_failure"


def clean_traceback_text(traceback_text: str) -> str:
    """Strip ANSI escapes and normalize traceback lines for model prompts."""
    text = traceback_text or ""
    ansi_escape = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")
    text = ansi_escape.sub("", text)
    lines = [ln.rstrip() for ln in text.splitlines()]
    # Keep all non-empty lines so model can see full context without terminal noise.
    cleaned = "\n".join([ln for ln in lines if ln.strip()])
    return cleaned or "No traceback available."


def build_error_brief(traceback_text: str, error_explanation: str = "", max_chars: int = 320) -> str:
    """Build compact error context: final line + optional short hint."""
    line = extract_failure_signature(traceback_text)
    hint = (error_explanation or "").strip()
    if len(hint) > max_chars:
        hint = hint[:max_chars].rstrip() + "..."

    if hint and hint != line:
        return f"{line}\nHint: {hint}"
    return line


def push_route(state: RepairState, node_name: str) -> None:
    """Record that we visited a node (for debugging/visualization)."""
    state["route_history"].append(node_name)

### 7) Configure models, RAG, and helper pipelines
```
# =============================================================================
# model clients + LCEL chains + RAG pipeline configuration
# =============================================================================
# Goal: Configure local/remote models and retrieval helpers for repair strategies.
# Output: Callable wrappers for SFT, reflection, external expert, and RAG stages.
```

In [78]:
# =============================================================================
# Configuration: local llama-server + OpenRouter (LangSmith traced)
# =============================================================================

# ----------------------------
# Local llama-server (CPU)
# ----------------------------
# `model` is the string sent in the OpenAI-compatible request body.
# llama.cpp does not require an API key; this is only to satisfy clients.
LLAMA_SERVER_URL = os.getenv("LLAMA_SERVER_URL", "http://127.0.0.1:8081/v1").strip()
LLAMA_OPENAI_MODEL = os.getenv("LLAMA_OPENAI_MODEL", "local-gguf").strip()

# ----------------------------
# OpenRouter (remote models)
# ----------------------------
OPENROUTER_API_KEY = (os.getenv("OPENROUTER_API_KEY") or "").strip() or None
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1").strip()
OPENROUTER_REFLECTION_MODEL = os.getenv("OPENROUTER_REFLECTION_MODEL", "anthropic/claude-sonnet-4.6").strip()
OPENROUTER_EXTERNAL_MODEL = os.getenv("OPENROUTER_EXTERNAL_MODEL", "anthropic/claude-opus-4.6").strip()
OPENROUTER_SUMMARIZER_MODEL = os.getenv("OPENROUTER_SUMMARIZER_MODEL", "openai/gpt-oss-120b").strip()

# Optional but recommended by OpenRouter for attribution/rate-limit policy
OPENROUTER_HTTP_REFERER = os.getenv("OPENROUTER_HTTP_REFERER", "").strip()
OPENROUTER_X_TITLE = os.getenv("OPENROUTER_X_TITLE", "MentorApp").strip()

OPENROUTER_HEADERS = {
    k: v
    for k, v in {
        "HTTP-Referer": OPENROUTER_HTTP_REFERER,
        "X-Title": OPENROUTER_X_TITLE,
    }.items()
    if v
}

# ----------------------------
# Runtime settings
# ----------------------------
# Increased defaults to reduce cut-off responses on long fixes.
HTTP_TIMEOUT_SECONDS = int(os.getenv("HTTP_TIMEOUT_SECONDS", "120"))
MAX_GENERATION_TOKENS = int(os.getenv("MAX_GENERATION_TOKENS", "3000"))

# ----------------------------
# ChromaDB & RAG configuration
# ----------------------------
CHROMA_DIR = str(Path("VectorDB/chroma_library_docs").resolve())
COLLECTION_NAME = "library_docs"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL = "BAAI/bge-reranker-base"

N_RETRIEVE = 10
N_RERANK = 3
MAX_QUERY_LEN = 500
MIN_RERANKER_SCORE = 0.0

_chroma_collection = None
_reranker = None

# =============================================================================
# LangChain model clients
# =============================================================================

def get_sft_llm() -> ChatOpenAI:
    """Local GGUF model served by llama.cpp `llama-server`."""
    return ChatOpenAI(
        base_url=LLAMA_SERVER_URL,
        api_key="llama.cpp",  # not used by llama-server
        model=LLAMA_OPENAI_MODEL,
        temperature=0.1,
        max_tokens=MAX_GENERATION_TOKENS,
        timeout=HTTP_TIMEOUT_SECONDS,
    )

def get_reflection_llm() -> ChatOpenAI:
    """OpenRouter model used for critique / diagnosis."""
    return ChatOpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=OPENROUTER_API_KEY,
        default_headers=OPENROUTER_HEADERS,
        model=OPENROUTER_REFLECTION_MODEL,
        temperature=0.1,
        max_tokens=700,
        timeout=HTTP_TIMEOUT_SECONDS,
    )


def get_external_llm() -> ChatOpenAI:
    """OpenRouter model used as a stronger external repair agent."""
    return ChatOpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=OPENROUTER_API_KEY,
        default_headers=OPENROUTER_HEADERS,
        model=OPENROUTER_EXTERNAL_MODEL,
        temperature=0.1,
        max_tokens=MAX_GENERATION_TOKENS,
        timeout=HTTP_TIMEOUT_SECONDS,
    )


def get_summarizer_llm() -> ChatOpenAI:
    """OpenRouter model used to summarize retrieved docs into bullet hints."""
    return ChatOpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=OPENROUTER_API_KEY,
        default_headers=OPENROUTER_HEADERS,
        model=OPENROUTER_SUMMARIZER_MODEL,
        temperature=0.0,
        max_tokens=400,
        timeout=HTTP_TIMEOUT_SECONDS,
    )


# =============================================================================
# LCEL Chains for RAG and Code Repair (LangSmith Traced)
# =============================================================================

# --- SFT Code Repair Chain ---
SFT_REPAIR_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a code repair model. Apply minimal edits only.
Fix code so it compiles and runs. Do not rewrite unrelated parts.
Return ONLY the corrected Python code wrapped in <correct_code> tags."""),
    ("human", """{context}

CURRENT_CODE:
{code}

<correct_code>""")
])

def create_sft_repair_chain():
    """Create LCEL chain for SFT code repair with stop token."""
    llm = get_sft_llm()
    return SFT_REPAIR_PROMPT | llm.bind(stop=["</correct_code>"]) | StrOutputParser()


# --- RAG Summarization Chain ---
RAG_SUMMARIZE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a concise technical summariser.
Given documentation snippets and an error message, produce a short list of 
actionable bullet-point hints that are DIRECTLY relevant to fixing the error.

Rules:
- Return ONLY bullet points (lines starting with "- ").
- Maximum 3 bullets, each ≤ 1 sentence.
- Each bullet MUST address the specific error shown, not general advice.
- Focus on the API signature, correct parameter names, or usage pattern that fixes the error.
- Do NOT include code blocks or examples.
- If the docs are NOT relevant to the error, write 2 short hints from your own knowledge."""),
    ("human", """Error: {error}

Documentation snippets:
{docs}""")
])

def create_rag_summarize_chain():
    """Create LCEL chain for RAG summarization."""
    llm = get_summarizer_llm()
    return RAG_SUMMARIZE_PROMPT | llm | StrOutputParser()


# --- Reflection Critic Chain ---
REFLECTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a strict Python code critic.
Return ONLY valid JSON with keys: diagnosis, hints, do_not_do, confidence.
- hints: concise bullet-like strings (list)
- do_not_do: constraints (list)  
- confidence: float in [0,1]
- Output raw JSON only (no markdown fences, no extra commentary).
Do not return repaired code."""),
    ("human", """Traceback:
{traceback}

Current code:
{code}

Recent attempts:
{attempts}""")
])

def create_reflection_chain():
    """Create LCEL chain for reflection/critic model."""
    llm = get_reflection_llm()
    return REFLECTION_PROMPT | llm | StrOutputParser()


# --- External Expert Chain ---
EXTERNAL_EXPERT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a careful Python repair assistant.
Return only corrected code wrapped in <correct_code> tags.
Keep edits as small as possible while fixing the failure."""),
    ("human", """TRACEBACK:
{traceback}

CURRENT_CODE:
{code}

<correct_code>""")
])

def create_external_expert_chain():
    """Create LCEL chain for external expert model with stop token."""
    llm = get_external_llm()
    return EXTERNAL_EXPERT_PROMPT | llm.bind(stop=["</correct_code>"]) | StrOutputParser()


# =============================================================================
# ChromaDB & Cross-Encoder Setup
# =============================================================================

def _get_chroma_collection():
    """Lazy-load ChromaDB collection."""
    global _chroma_collection
    if _chroma_collection is None:
        embedding_fn = SentenceTransformerEmbeddingFunction(
            model_name=EMBEDDING_MODEL,
            device="cpu",
            normalize_embeddings=True,
        )
        client = chromadb.PersistentClient(path=CHROMA_DIR)
        _chroma_collection = client.get_collection(
            name=COLLECTION_NAME,
            embedding_function=embedding_fn,
        )
        print(f"[ChromaDB] Loaded '{COLLECTION_NAME}' with {_chroma_collection.count()} docs")
    return _chroma_collection


def _get_reranker():
    """Lazy-load cross-encoder reranker."""
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder(RERANKER_MODEL, max_length=512)
        print(f"[Reranker] Loaded {RERANKER_MODEL}")
    return _reranker


# =============================================================================
# Helper Functions
# =============================================================================

def _extract_current_code_from_prompt(prompt: str) -> str:
    """Extract the CURRENT_CODE section from a repair prompt."""
    marker = "CURRENT_CODE:\n"
    idx = prompt.find(marker)
    return prompt[idx + len(marker):].strip() if idx != -1 else prompt


def _extract_code_block(text: str) -> str:
    """Extract code from various formats: <correct_code> tags, markdown fences, or raw."""
    if not text:
        return text
    
    # First try <correct_code> tags
    tag_match = re.search(r"<correct_code>(.*?)(?:</correct_code>|$)", text, re.DOTALL)
    if tag_match:
        return tag_match.group(1).strip()
    
    # Then try markdown fences
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        return fenced[0].strip()
    
    return text.strip()


def _heuristic_minimal_fix(code: str, context: str) -> str:
    """Fallback fixer when API calls fail."""
    fixed = code.replace("pritn(", "print(")
    fixed_lines = []
    block_keywords = ("def ", "if ", "for ", "while ", "class ", "elif ", "else", "try", "except", "finally")
    for line in fixed.splitlines():
        stripped = line.strip()
        needs_colon = (
            any(stripped.startswith(kw) for kw in block_keywords)
            and not stripped.endswith(":")
            and not stripped.startswith("#")
        )
        fixed_lines.append(line + ":" if needs_colon else line)
    fixed = "\n".join(fixed_lines)
    if "NameError: name 'np' is not defined" in context and "import numpy as np" not in fixed:
        fixed = "import numpy as np\n" + fixed
    return fixed


# =============================================================================
# Model Calling Functions (Using LCEL Chains)
# =============================================================================

def call_sft_model(prompt: str) -> str:
    """Call SFT repair model using LCEL chain (LangSmith traced)."""
    original_code = _extract_current_code_from_prompt(prompt)
    
    try:
        # Parse prompt to extract context and code
        context = ""
        code = original_code
        
        if "TRACEBACK:" in prompt:
            parts = prompt.split("CURRENT_CODE:")
            context = parts[0] if len(parts) > 1 else ""
            code = parts[1].strip() if len(parts) > 1 else original_code
        elif "HINTS FROM DOCUMENTATION:" in prompt:
            parts = prompt.split("CURRENT_CODE:")
            context = parts[0] if len(parts) > 1 else ""
            code = parts[1].strip() if len(parts) > 1 else original_code
        elif "REFLECTION_FEEDBACK:" in prompt:
            parts = prompt.split("CURRENT_CODE:")
            context = parts[0] if len(parts) > 1 else ""
            code = parts[1].strip() if len(parts) > 1 else original_code
        
        chain = create_sft_repair_chain()
        result = chain.invoke({"context": context, "code": code})
        candidate = _extract_code_block(result)
        
        if candidate:
            return candidate
            
    except Exception as e:
        print(f"[call_sft_model] LCEL chain failed, using fallback: {e}")
    
    return _heuristic_minimal_fix(original_code, prompt)


def call_reflection_model(current_code: str, traceback_text: str, attempt_history: list[dict]) -> dict[str, Any]:
    """Call reflection/critic model using LCEL chain (LangSmith traced)."""
    fallback = {
        "diagnosis": extract_failure_signature(traceback_text),
        "hints": ["Focus on the top traceback line.", "Apply the smallest change that fixes the failure."],
        "do_not_do": ["No full rewrites"],
        "confidence": 0.35,
    }
    
    if not OPENROUTER_API_KEY:
        return fallback
    
    try:
        chain = create_reflection_chain()
        result = chain.invoke({
            "traceback": traceback_text,
            "code": current_code,
            "attempts": str(attempt_history[-3:])
        })
        raw = (result or "").strip()
        if not raw:
            return fallback

        # Some models still return fenced JSON or extra prose.
        raw = re.sub(r"^```(?:json)?\\n|\\n```$", "", raw.strip(), flags=re.IGNORECASE)
        match = re.search(r"\\{.*\\}", raw, flags=re.DOTALL)
        json_text = match.group(0) if match else raw

        parsed = json.loads(json_text)
        if not isinstance(parsed, dict):
            return fallback

        # Fill missing fields defensively to keep downstream nodes stable.
        parsed.setdefault("diagnosis", fallback["diagnosis"])
        parsed.setdefault("hints", fallback["hints"])
        parsed.setdefault("do_not_do", fallback["do_not_do"])
        parsed.setdefault("confidence", fallback["confidence"])
        if not isinstance(parsed.get("hints"), list):
            parsed["hints"] = fallback["hints"]
        if not isinstance(parsed.get("do_not_do"), list):
            parsed["do_not_do"] = fallback["do_not_do"]
        return parsed
        
    except Exception as e:
        print(f"[call_reflection_model] LCEL chain failed: {e}")
        return fallback


def call_external_model(prompt: str) -> str:
    """Call external expert model using LCEL chain (LangSmith traced)."""
    original_code = _extract_current_code_from_prompt(prompt)
    
    if not OPENROUTER_API_KEY:
        return _heuristic_minimal_fix(original_code, prompt)
    
    try:
        # Parse traceback from prompt
        traceback_text = ""
        if "TRACEBACK:" in prompt:
            start = prompt.find("TRACEBACK:") + len("TRACEBACK:")
            end = prompt.find("CURRENT_CODE:")
            traceback_text = prompt[start:end].strip() if end > start else prompt[start:].strip()
        
        chain = create_external_expert_chain()
        result = chain.invoke({
            "traceback": traceback_text,
            "code": original_code
        })
        candidate = _extract_code_block(result)
        
        if candidate:
            return candidate
            
    except Exception as e:
        print(f"[call_external_model] LCEL chain failed: {e}")
    
    return _heuristic_minimal_fix(original_code, prompt)


# =============================================================================
# RAG Pipeline: Retrieve -> Rerank -> Summarize (with LCEL Chain)
# =============================================================================

def retrieve_from_vector_db(query: str, n_results: int = N_RETRIEVE) -> list[tuple[str, dict]]:
    """Stage 1: Retrieve documents from ChromaDB using bi-encoder."""
    try:
        collection = _get_chroma_collection()
        query_text = query[:MAX_QUERY_LEN]
        
        results = collection.query(
            query_texts=[query_text],
            n_results=n_results,
            include=["documents", "metadatas"],
        )
        
        docs = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        
        candidates = list(zip(docs, metadatas))
        if candidates:
            print(f"[RAG Stage 1] Retrieved {len(candidates)} docs for: {query_text[:50]}...")
        return candidates
        
    except Exception as e:
        print(f"[retrieve_from_vector_db] ChromaDB query failed: {e}")
        return []


def rerank_docs(query: str, candidates: list[tuple[str, dict]], top_k: int = N_RERANK) -> list[tuple[float, str, dict]]:
    """Stage 2: Rerank candidates using cross-encoder."""
    if not candidates:
        return []
    
    try:
        reranker = _get_reranker()
        query_text = query[:MAX_QUERY_LEN]
        
        pairs = [(query_text, doc) for doc, _ in candidates]
        scores = reranker.predict(pairs)
        
        scored = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        
        result = [
            (float(score), doc, meta)
            for score, (doc, meta) in scored
            if score > MIN_RERANKER_SCORE
        ][:top_k]
        
        if result:
            print(f"[RAG Stage 2] Reranked to top {len(result)} docs (scores: {[f'{s:.2f}' for s, _, _ in result]})")
        return result
        
    except Exception as e:
        print(f"[rerank_docs] Reranking failed: {e}")
        return [(0.5, doc, meta) for doc, meta in candidates[:top_k]]


def summarize_docs_to_hints(reranked_docs: list[tuple[float, str, dict]], error_text: str) -> str:
    """Stage 3: Summarize reranked docs + error using LCEL chain (LangSmith traced)."""
    if not reranked_docs:
        return ""
    
    # Format docs for summarizer
    docs_text = "\n---\n".join([
        f"[{meta.get('library', 'unknown')} v{meta.get('version', '?')} score={score:.2f}]\n{doc}"
        for score, doc, meta in reranked_docs
    ])
    
    # Extract the key error line
    error_lines = [l.strip() for l in error_text.splitlines() if l.strip()]
    error_line = error_lines[-1] if error_lines else error_text[:200]
    
    if not OPENROUTER_API_KEY:
        # Fallback: simple extraction without LLM
        bullets = []
        for _, doc, _ in reranked_docs[:3]:
            first_line = doc.strip().splitlines()[0] if doc.strip() else ""
            if first_line:
                bullets.append(f"- {first_line[:150]}")
        return "\n".join(bullets[:3])
    
    try:
        chain = create_rag_summarize_chain()
        content = chain.invoke({"error": error_line, "docs": docs_text})
        
        # Keep only lines that look like bullets
        bullets = [
            line.strip()
            for line in content.split("\n")
            if line.strip().startswith("- ")
        ][:3]
        
        hints = "\n".join(bullets) if bullets else content.strip()
        print(f"[RAG Stage 3] Generated {len(bullets)} bullet hints ({len(hints)} chars)")
        return hints
        
    except Exception as e:
        print(f"[summarize_docs_to_hints] LCEL chain failed: {e}")
        return ""


def call_summary_model(chunks: list[str]) -> str:
    """Legacy wrapper for simple summarization."""
    bullets = []
    for chunk in chunks[:6]:
        first_line = chunk.strip().splitlines()[0] if chunk.strip() else ""
        if first_line:
            bullets.append(f"- {first_line[:180]}")
    return "\n".join(bullets) if bullets else "- No helpful context found."


# --- Web Search (DuckDuckGo) ---

def web_search(query: str, max_results: int = 5) -> list[str]:
    """Search the web using DuckDuckGo for Python error solutions."""
    if not DDGS_AVAILABLE:
        return [
            "Official Python traceback guide: focus on final exception line and originating frame.",
            "Best practice: apply smallest diff that resolves failing tests/runtime.",
        ]
    
    try:
        search_query = f"python {query} fix solution"
        with DDGS() as ddgs:
            results = list(ddgs.text(search_query, max_results=max_results))
        
        snippets = []
        for r in results:
            title = r.get("title", "")
            body = r.get("body", "")
            snippet = f"{title}: {body}" if title else body
            if snippet:
                snippets.append(snippet[:300])
        
        if snippets:
            print(f"[WebSearch] Found {len(snippets)} results for: {query[:50]}...")
        return snippets
        
    except Exception as e:
        print(f"[web_search] DuckDuckGo search failed: {e}")
        return [
            "Official Python traceback guide: focus on final exception line and originating frame.",
            "Best practice: apply smallest diff that resolves failing tests/runtime.",
        ]


# =============================================================================
# Error Classification Function
# =============================================================================

def extract_error_details(traceback_text: str) -> tuple[str, str, str]:
    """Extract (error_type, error_message, full_error_line) from traceback text."""
    lines = [line.strip() for line in traceback_text.splitlines() if line.strip()]
    if not lines:
        return "UnknownError", "", ""

    # Find the last line that looks like `SomeError: message`.
    for line in reversed(lines):
        m = re.match(r"^([A-Za-z_][A-Za-z0-9_]*(?:Error|Exception)):\\s*(.*)$", line)
        if m:
            err_type = m.group(1)
            err_msg = m.group(2).strip()
            return err_type, err_msg, line

    # Fallback to the last non-empty line.
    tail = lines[-1]
    return "UnknownError", tail, tail


def classify_error(traceback_text: str, check_result: dict) -> dict[str, str]:
    """
    Classify the error type for intelligent routing.
    
    Returns dict with:
    - category: routing category
    - error_type: extracted exception type (e.g., ValueError)
    - error_explanation: extracted exception message
    - error_line: most relevant traceback tail line

    Categories:
    - syntax_error: SyntaxError, IndentationError
    - name_error: NameError, AttributeError (typos)
    - timeout: Execution timeout
    - api_library_error: ImportError, ModuleNotFoundError, API-related errors
    - local_reasoning_error: ValueError, TypeError, shape errors, logic errors
    """
    error_type, error_msg, error_line = extract_error_details(traceback_text)
    msg_lower = (error_msg or error_line or "").lower()
    
    # If compile failed, treat as syntax class by design.
    if not check_result.get("compile_ok", True):
        return {
            "category": "syntax_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }
    
    # Timeout
    if "timeout" in msg_lower or "timeoutexpired" in msg_lower:
        return {
            "category": "timeout",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }
    
    # Name/Attribute errors (usually typos)
    if error_type.lower() == "nameerror":
        return {
            "category": "name_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }
    
    # API/Library errors (RAG can help)
    api_indicators = [
        "importerror", "modulenotfounderror", "attributeerror",
        "deprecat", "no module named", "cannot import",
        "unexpected keyword argument", "positional argument",
        "got an unexpected", "missing required"
    ]
    if error_type in {"ImportError", "ModuleNotFoundError", "AttributeError"} or any(ind in msg_lower for ind in api_indicators):
        return {
            "category": "api_library_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }
    
    # Shape/dimension/type errors (need reasoning)
    reasoning_indicators = [
        "valueerror", "typeerror", "indexerror", "keyerror",
        "shape", "dimension", "broadcast", "mismatch",
        "cannot convert", "invalid"
    ]
    if error_type in {"ValueError", "TypeError", "IndexError", "KeyError", "AssertionError"} or any(ind in msg_lower for ind in reasoning_indicators):
        return {
            "category": "local_reasoning_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }
    
    # Default: treat as API error (RAG might help)
    return {
        "category": "api_library_error",
        "error_type": error_type,
        "error_explanation": error_msg,
        "error_line": error_line,
    }


print("✓ Configuration loaded | LangchainOpenAI models configured for LangSmith tracing")
print(f"[Config] MAX_GENERATION_TOKENS={MAX_GENERATION_TOKENS}, HTTP_TIMEOUT_SECONDS={HTTP_TIMEOUT_SECONDS}")

✓ Configuration loaded | LangchainOpenAI models configured for LangSmith tracing
[Config] MAX_GENERATION_TOKENS=3000, HTTP_TIMEOUT_SECONDS=120


### 8) Implement validation node
```
# =============================================================================
# validation node: compile + runtime checks
# =============================================================================
# Goal: Verify candidate code and capture traceback/failure signatures.
# Output: Structured check results used by diagnosis and strategy routing.
```

In [106]:
# =============================================================================
# Core Validation Node: run_checks
# =============================================================================

# Runtime check timeout (seconds) for candidate execution.
# Increase this for heavier scripts (statsmodels training, plotting, etc.).
EXECUTION_TIMEOUT_SECONDS = int(os.getenv("EXECUTION_TIMEOUT_SECONDS", "45"))

def run_checks(state: RepairState) -> RepairState:
    """
    Execute code validation: compile check + runtime execution.
    Updates state with check_result, traceback, and failure_signature.
    """
    push_route(state, "run_checks")
    code = state["current_code"]

    result: dict[str, Any] = {
        "passed": False,
        "compile_ok": False,
        "runtime_ok": False,
        "stdout": "",
        "stderr": "",
        "traceback": "",
    }

    # Step 1: Compile check
    try:
        compile(code, "<candidate_code>", "exec")
        result["compile_ok"] = True
    except Exception as e:
        trace = "".join(tb.format_exception(e))
        result["traceback"] = trace
        state["traceback"] = trace
        state["check_result"] = result
        state["failure_signature"] = extract_failure_signature(trace)
        return state

    # Step 2: Runtime execution
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as f:
            f.write(code)
            temp_path = f.name

        proc = subprocess.run(
            [sys.executable, temp_path],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=EXECUTION_TIMEOUT_SECONDS,
        )
        result["stdout"] = proc.stdout
        result["stderr"] = proc.stderr

        if proc.returncode == 0:
            result["runtime_ok"] = True
            result["passed"] = True
            state["traceback"] = ""
            state["failure_signature"] = ""
        else:
            trace = proc.stderr or "Runtime failed without stderr output."
            result["traceback"] = trace
            state["traceback"] = trace
            state["failure_signature"] = extract_failure_signature(trace)
    except subprocess.TimeoutExpired as e:
        timeout_tb = f"Execution timeout ({EXECUTION_TIMEOUT_SECONDS}s): {e}"
        result["traceback"] = timeout_tb
        state["traceback"] = timeout_tb
        state["failure_signature"] = "TimeoutExpired"
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

    state["check_result"] = result
    return state


def _record_attempt(state: RepairState, route: str, before_code: str, after_code: str, prompt: str) -> None:
    """Record an attempt in history and track meaningful changes."""
    changed = code_changed_meaningfully(before_code, after_code)
    state["attempt_count"] += 1
    state["attempt_history"].append({
        "attempt": state["attempt_count"],
        "route": route,
        "changed_meaningfully": changed,
        "timestamp": now_iso(),
        "prompt_preview": prompt[:220],
    })

    # Track stagnation (no meaningful changes)
    state["no_meaningful_change_count"] = 0 if changed else state["no_meaningful_change_count"] + 1

print(f"[Config] EXECUTION_TIMEOUT_SECONDS={EXECUTION_TIMEOUT_SECONDS}")

[Config] EXECUTION_TIMEOUT_SECONDS=45


### 9) Implement core repair and routing decisions
```
# =============================================================================
# core repair nodes + diagnosis + strategy selection
# =============================================================================
# Goal: Run initial fixes, classify failures, and choose the next strategy.
# Includes: Same-error detection for reflection/RAG to external escalation.
```

In [95]:
# =============================================================================
# SFT Repair Nodes & Flow Control
# =============================================================================

def attempt_sft_initial(state: RepairState) -> RepairState:
    """First repair attempt: plain SFT model without additional context."""
    push_route(state, "attempt_sft_initial")
    before = state["current_code"]
    prompt = (
        "You are a code repair model. Apply minimal edits only.\n"
        "Fix code so it compiles and runs. Do not rewrite unrelated parts.\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_sft_model(prompt)
    state["current_code"] = candidate
    _record_attempt(state, "attempt_sft_initial", before, candidate, prompt)
    return state


def attempt_sft_with_traceback(state: RepairState) -> RepairState:
    """Traceback-guided repair attempt."""
    push_route(state, "attempt_sft_with_traceback")
    if not state.get("initial_traceback"):
        state["initial_traceback"] = state.get("traceback", "")
    before = state["current_code"]
    tb_brief = build_error_brief(
        traceback_text=state.get("traceback", ""),
        error_explanation=state.get("error_explanation", ""),
    )
    prompt = (
        "Repair code using concise error evidence. Apply minimal edits only.\n"
        f"TRACEBACK:\n{tb_brief}\n\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_sft_model(prompt)
    state["used_traceback"] = True
    state["current_code"] = candidate
    _record_attempt(state, "attempt_sft_with_traceback", before, candidate, prompt)
    return state


def diagnose_failure(state: RepairState) -> RepairState:
    """
    Analyze failure and classify error type for intelligent routing.
    Sets error_category and checks for stopping conditions.
    """
    push_route(state, "diagnose_failure")

    classify_tb = state.get("traceback", "")
    classify_check_result = state.get("check_result", {})
    if state.get("used_traceback") and state.get("attempt_count", 0) == 1 and state.get("initial_traceback"):
        classify_tb = state.get("initial_traceback", "")
        classify_check_result = {"compile_ok": True}
    classification = classify_error(
        traceback_text=classify_tb,
        check_result=classify_check_result,
    )
    error_category = classification.get("category", "api_library_error")
    state["error_category"] = error_category
    state["error_type"] = classification.get("error_type", "UnknownError")
    state["error_explanation"] = classification.get("error_explanation", "")
    print(f"[diagnose_failure] Error classified as: {error_category} | {state['error_type']}: {state['error_explanation']}")

    signature = state.get("failure_signature", "")
    previous_signature = state.get("previous_failure_signature", "")
    if signature and previous_signature and signature == previous_signature:
        state["repeated_failure_count"] += 1
    else:
        state["repeated_failure_count"] = 0
    if signature:
        state["previous_failure_signature"] = signature

    if state["attempt_count"] >= state["max_attempts"]:
        state["should_stop"] = True
        state["stop_reason"] = "max_attempts_reached"
    elif state["no_meaningful_change_count"] >= 2:
        if state["used_reflection"] and not state["used_external"]:
            state["should_stop"] = False
            state["stop_reason"] = ""
        else:
            state["should_stop"] = True
            state["stop_reason"] = "no_meaningful_change"

    return state


def choose_next_strategy(state: RepairState) -> RepairState:
    """
    Select next strategy by error class.

    Rule: RAG is ONLY for api_library_error.
    Non-API errors use traceback/reflection first, then external.
    """
    push_route(state, "choose_next_strategy")

    if state["should_stop"]:
        if state.get("stop_reason") == "no_meaningful_change" and state["used_reflection"] and not state["used_external"]:
            state["should_stop"] = False
            state["stop_reason"] = ""
        else:
            state["next_strategy"] = "finalize_failure"
            return state

    error_category = state.get("error_category", "api_library_error")

    # API/library only: RAG allowed.
    if error_category == "api_library_error":
        if not state["used_rag"]:
            state["next_strategy"] = "local_rag"
            print(f"[choose_strategy] {error_category} -> RAG pipeline")
        elif not state["used_traceback"]:
            state["next_strategy"] = "traceback_sft"
            print(f"[choose_strategy] {error_category} -> Traceback-guided SFT")
        elif not state["used_reflection"]:
            state["next_strategy"] = "reflection_critic"
            print(f"[choose_strategy] {error_category} -> Reflection")
        elif not state["used_external"]:
            state["next_strategy"] = "external_expert"
            print(f"[choose_strategy] {error_category} -> External expert")
        else:
            state["next_strategy"] = "finalize_failure"
            state["should_stop"] = True
            state["stop_reason"] = "all_strategies_exhausted"
        return state

    # Non-API classes: no RAG.
    if error_category in ("syntax_error", "name_error", "timeout"):
        if not state["used_traceback"]:
            state["next_strategy"] = "traceback_sft"
            print(f"[choose_strategy] {error_category} -> Traceback-guided SFT")
        elif not state["used_reflection"]:
            state["next_strategy"] = "reflection_critic"
            print(f"[choose_strategy] {error_category} -> Reflection")
        elif not state["used_external"]:
            state["next_strategy"] = "external_expert"
            print(f"[choose_strategy] {error_category} -> External expert")
        else:
            state["next_strategy"] = "finalize_failure"
            state["should_stop"] = True
            state["stop_reason"] = "all_strategies_exhausted"
        return state

    # local_reasoning_error or unknown: reflection first, then traceback, then external.
    if not state["used_reflection"]:
        state["next_strategy"] = "reflection_critic"
        print(f"[choose_strategy] {error_category} -> Reflection")
    elif not state["used_traceback"]:
        state["next_strategy"] = "traceback_sft"
        print(f"[choose_strategy] {error_category} -> Traceback-guided SFT")
    elif not state["used_external"]:
        state["next_strategy"] = "external_expert"
        print(f"[choose_strategy] {error_category} -> External expert")
    else:
        state["next_strategy"] = "finalize_failure"
        state["should_stop"] = True
        state["stop_reason"] = "all_strategies_exhausted"

    return state

### 10) Implement strategy nodes
```
# =============================================================================
# strategy branches: RAG, reflection, external expert
# =============================================================================
# Goal: Implement specialized repair routes after strategy selection.
# Behavior: Each branch regenerates code, then returns to validation.
```

In [84]:
# =============================================================================
# Strategy Nodes: RAG, Reflection, External Expert
# =============================================================================

# --- RAG Pipeline (Retrieve → Rerank → Summarize) ---

def retrieve_local_docs(state: RepairState) -> RepairState:
    """
    RAG Stage 1: Retrieve 10 documents from ChromaDB using bi-encoder.
    """
    push_route(state, "retrieve_local_docs")
    query = state.get("traceback") or state.get("failure_signature") or "python error fix"
    
    # Stage 1: Retrieve candidates
    candidates = retrieve_from_vector_db(query, n_results=N_RETRIEVE)
    state["local_docs"] = candidates  # Store as list of (doc, meta) tuples
    state["used_rag"] = True
    return state


def assess_local_context(state: RepairState) -> RepairState:
    """
    RAG Stage 2: Rerank candidates using cross-encoder to get top 3.
    Evaluate quality based on reranker scores.
    """
    push_route(state, "assess_local_context")
    
    candidates = state.get("local_docs", [])
    if not candidates:
        state["local_context_quality"] = "weak"
        return state
    
    # Stage 2: Rerank to top 3
    query = state.get("traceback") or state.get("failure_signature") or ""
    reranked = rerank_docs(query, candidates, top_k=N_RERANK)
    
    # Store reranked docs for summarization
    state["local_docs"] = reranked  # Now list of (score, doc, meta) tuples
    
    # Assess quality based on reranker scores
    if reranked and len(reranked) >= 2:
        avg_score = sum(s for s, _, _ in reranked) / len(reranked)
        state["local_context_quality"] = "good" if avg_score > 0.3 else "weak"
    else:
        state["local_context_quality"] = "weak"
    
    return state


def web_search_docs(state: RepairState) -> RepairState:
    """Augment with web search when local context is weak."""
    push_route(state, "web_search_docs")
    query = state.get("failure_signature") or "python runtime error fix"
    state["web_docs"] = web_search(query)
    state["used_web"] = True
    return state


def summarize_context(state: RepairState) -> RepairState:
    """
    RAG Stage 3: Summarize top 3 reranked docs + traceback into max 3 bullet hints.
    """
    push_route(state, "summarize_context")
    
    reranked_docs = state.get("local_docs", [])
    traceback_text = state.get("traceback", "")
    
    # Check if we have reranked docs (tuple format with scores)
    if reranked_docs and isinstance(reranked_docs[0], tuple) and len(reranked_docs[0]) == 3:
        # Use the full summarization pipeline with LLM
        hints = summarize_docs_to_hints(reranked_docs, traceback_text)
    else:
        # Fallback: combine with web docs and use simple extraction
        web_docs = state.get("web_docs", [])
        all_chunks = []
        
        # Handle different formats of local_docs
        if reranked_docs:
            if isinstance(reranked_docs[0], tuple):
                all_chunks.extend([doc for doc, _ in reranked_docs])
            else:
                all_chunks.extend(reranked_docs)
        all_chunks.extend(web_docs)
        
        hints = call_summary_model(all_chunks)
    
    state["summarized_hints"] = hints
    return state


def attempt_sft_with_rag(state: RepairState) -> RepairState:
    """Repair attempt using RAG-retrieved context (bullet hints)."""
    push_route(state, "attempt_sft_with_rag")
    before = state["current_code"]
    tb_brief = build_error_brief(
        traceback_text=state.get("traceback", ""),
        error_explanation=state.get("error_explanation", ""),
    )
    prompt = (
        "Repair code with these documentation hints. Minimal edits only.\n"
        f"HINTS FROM DOCUMENTATION:\n{state.get('summarized_hints', '')}\n\n"
        f"TRACEBACK:\n{tb_brief}\n\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_sft_model(prompt)
    state["current_code"] = candidate
    _record_attempt(state, "attempt_sft_with_rag", before, candidate, prompt)
    return state


# --- Reflection Pipeline ---

def reflection_critic(state: RepairState) -> RepairState:
    """Get structured debugging feedback from reflection model."""
    push_route(state, "reflection_critic")
    feedback = call_reflection_model(
        current_code=state["current_code"],
        traceback_text=state.get("traceback", ""),
        attempt_history=state.get("attempt_history", []),
    )
    state["reflection_feedback"] = feedback
    state["used_reflection"] = True
    return state


def attempt_sft_with_reflection(state: RepairState) -> RepairState:
    """Repair attempt using reflection feedback."""
    push_route(state, "attempt_sft_with_reflection")
    before = state["current_code"]
    feedback = state.get("reflection_feedback", {})
    tb_brief = build_error_brief(
        traceback_text=state.get("traceback", ""),
        error_explanation=state.get("error_explanation", ""),
    )
    prompt = (
        "Repair code using reflection hints. Do not produce full rewrite. Minimal edits only.\n"
        f"REFLECTION_FEEDBACK:\n{feedback}\n\n"
        f"TRACEBACK:\n{tb_brief}\n\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_sft_model(prompt)
    state["current_code"] = candidate
    _record_attempt(state, "attempt_sft_with_reflection", before, candidate, prompt)
    return state


# --- External Expert ---

def external_expert_repair(state: RepairState) -> RepairState:
    """Final fallback: use external expert model."""
    push_route(state, "external_expert_repair")
    before = state["current_code"]
    reflection_feedback = state.get("reflection_feedback", {})
    summarized_hints = state.get("summarized_hints", "")
    tb_brief = build_error_brief(
        traceback_text=state.get("traceback", ""),
        error_explanation=state.get("error_explanation", ""),
    )
    prompt = (
        "Final fallback repair. Keep edits as small as possible while fixing failure.\n"
        f"TRACEBACK:\n{tb_brief}\n\n"
        f"REFLECTION_HINTS:\n{reflection_feedback}\n\n"
        f"RAG_HINTS:\n{summarized_hints}\n\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_external_model(prompt)
    state["used_external"] = True
    state["current_code"] = candidate
    _record_attempt(state, "external_expert_repair", before, candidate, prompt)
    return state

In [100]:
# Override: stricter SFT prompting policy for RAG vs non-RAG paths
def attempt_sft_with_traceback(state: RepairState) -> RepairState:
    """Traceback-guided repair attempt (non-RAG): full cleaned traceback."""
    push_route(state, "attempt_sft_with_traceback")
    if not state.get("initial_traceback"):
        state["initial_traceback"] = state.get("traceback", "")
    before = state["current_code"]
    cleaned_tb = clean_traceback_text(state.get("traceback", ""))
    prompt = (
        "Repair code using this cleaned traceback. Apply minimal edits only.\n"
        "Solve the error in the code based on the traceback details.\n"
        "Do not do a full rewrite.\n"
        f"TRACEBACK:\n{cleaned_tb}\n\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_sft_model(prompt)
    state["used_traceback"] = True
    state["current_code"] = candidate
    _record_attempt(state, "attempt_sft_with_traceback", before, candidate, prompt)
    return state


def attempt_sft_with_rag(state: RepairState) -> RepairState:
    """RAG-guided SFT attempt: enforce doc hints + concise error line."""
    push_route(state, "attempt_sft_with_rag")
    before = state["current_code"]
    tb_brief = build_error_brief(
        traceback_text=state.get("traceback", ""),
        error_explanation=state.get("error_explanation", ""),
    )
    prompt = (
        "Repair code using documentation hints as HARD constraints.\n"
        "You MUST follow every hint in HINTS FROM DOCUMENTATION.\n"
        "Base your fix on these hints and the error line below.\n"
        "Apply minimal edits only; do not rewrite unrelated code.\n"
        f"HINTS FROM DOCUMENTATION:\n{state.get('summarized_hints', '')}\n\n"
        f"TRACEBACK:\n{tb_brief}\n\n"
        f"CURRENT_CODE:\n{before}\n"
    )
    candidate = call_sft_model(prompt)
    state["current_code"] = candidate
    _record_attempt(state, "attempt_sft_with_rag", before, candidate, prompt)
    return state


print("[Override] sft_with_traceback now uses full cleaned traceback.")

[Override] sft_with_traceback now uses full cleaned traceback.


In [94]:
# Override: strict classifier guard so syntax never routes to RAG
def classify_error(traceback_text: str, check_result: dict) -> dict[str, str]:
    """Classify errors with hard guard: syntax signals always map to syntax_error."""
    error_type, error_msg, error_line = extract_error_details(traceback_text)
    msg_lower = (error_msg or error_line or "").lower()
    line_lower = (error_line or "").lower()
    err_lower = (error_type or "").lower()

    # Hard syntax guard regardless of compile_ok override state.
    if (
        not check_result.get("compile_ok", True)
        or "syntaxerror" in line_lower
        or "indentationerror" in line_lower
        or "syntaxerror" in msg_lower
        or "indentationerror" in msg_lower
        or err_lower in {"syntaxerror", "indentationerror"}
    ):
        return {
            "category": "syntax_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }

    if "timeout" in msg_lower or "timeoutexpired" in msg_lower:
        return {
            "category": "timeout",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }

    if err_lower == "nameerror":
        return {
            "category": "name_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }

    api_indicators = [
        "importerror", "modulenotfounderror", "attributeerror",
        "deprecat", "no module named", "cannot import",
        "unexpected keyword argument", "positional argument",
        "got an unexpected", "missing required"
    ]
    if error_type in {"ImportError", "ModuleNotFoundError", "AttributeError"} or any(ind in msg_lower for ind in api_indicators):
        return {
            "category": "api_library_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }

    reasoning_indicators = [
        "valueerror", "typeerror", "indexerror", "keyerror",
        "shape", "dimension", "broadcast", "mismatch",
        "cannot convert", "invalid"
    ]
    if error_type in {"ValueError", "TypeError", "IndexError", "KeyError", "AssertionError"} or any(ind in msg_lower for ind in reasoning_indicators):
        return {
            "category": "local_reasoning_error",
            "error_type": error_type,
            "error_explanation": error_msg,
            "error_line": error_line,
        }

    return {
        "category": "api_library_error",
        "error_type": error_type,
        "error_explanation": error_msg,
        "error_line": error_line,
    }

print("[Override] classify_error now hard-locks syntax signals to syntax_error.")

[Override] classify_error now hard-locks syntax signals to syntax_error.


### 11) Build the LangGraph workflow
```
# =============================================================================
# graph assembly + conditional routing edges
# =============================================================================
# Goal: Connect all nodes into one executable LangGraph workflow.
# Output: Compiled graph with deterministic transitions and stop terminals.
```

In [101]:
# =============================================================================
# Graph Construction & Routing Logic
# =============================================================================

def finalize_success(state: RepairState) -> RepairState:
    """Terminal node: code passed all checks."""
    push_route(state, "finalize_success")
    state["final_status"] = "success"
    state["final_code"] = state["current_code"]
    return state


def finalize_failure(state: RepairState) -> RepairState:
    """Terminal node: exhausted all strategies or hit stop condition."""
    push_route(state, "finalize_failure")
    state["final_status"] = "failure"
    state["final_code"] = state["current_code"]
    return state


# --- Conditional Edge Functions ---

def _route_after_checks(state: RepairState) -> str:
    """Route after run_checks: success, diagnose, or terminal after external."""
    if state["check_result"].get("passed", False):
        return "success"
    # External expert is the final repair attempt. If it fails, terminate.
    if state.get("used_external", False):
        return "external_failed"
    # Always diagnose first so strategy selection can choose the right branch early.
    return "diagnose"


def _route_after_diagnose(state: RepairState) -> str:
    """Route after diagnose: stop or continue to strategy selection."""
    return "finalize_failure" if state.get("should_stop", False) else "choose"


def _route_strategy(state: RepairState) -> str:
    """Route to the selected strategy."""
    return state.get("next_strategy", "finalize_failure")


def _route_local_context(state: RepairState) -> str:
    """Route based on local context quality: weak -> web search, else summarize."""
    return "web" if state.get("local_context_quality") == "weak" else "summarize"


def build_graph() -> StateGraph:
    """
    Construct the LangGraph workflow.

    Graph structure:
    run_checks -> diagnose_failure -> choose_next_strategy -> selected strategy
    selected strategy nodes return to run_checks until success/failure terminal.
    """
    builder = StateGraph(RepairState)

    # --- Add all nodes ---
    builder.add_node("attempt_sft_with_traceback", attempt_sft_with_traceback)
    builder.add_node("run_checks", run_checks)
    builder.add_node("diagnose_failure", diagnose_failure)
    builder.add_node("choose_next_strategy", choose_next_strategy)
    builder.add_node("retrieve_local_docs", retrieve_local_docs)
    builder.add_node("assess_local_context", assess_local_context)
    builder.add_node("web_search_docs", web_search_docs)
    builder.add_node("summarize_context", summarize_context)
    builder.add_node("attempt_sft_with_rag", attempt_sft_with_rag)
    builder.add_node("reflection_critic", reflection_critic)
    builder.add_node("attempt_sft_with_reflection", attempt_sft_with_reflection)
    builder.add_node("external_expert_repair", external_expert_repair)
    builder.add_node("finalize_success", finalize_success)
    builder.add_node("finalize_failure", finalize_failure)

    # --- Set entry point ---
    builder.set_entry_point("run_checks")

    # --- Direct edges: all repair nodes -> run_checks ---
    for node in [
        "attempt_sft_with_traceback",
        "attempt_sft_with_rag",
        "attempt_sft_with_reflection",
        "external_expert_repair",
    ]:
        builder.add_edge(node, "run_checks")

    # --- Conditional routing after run_checks ---
    builder.add_conditional_edges(
        "run_checks",
        _route_after_checks,
        {
            "success": "finalize_success",
            "diagnose": "diagnose_failure",
            "external_failed": "finalize_failure",
        },
    )

    # --- Conditional routing after diagnose ---
    builder.add_conditional_edges(
        "diagnose_failure",
        _route_after_diagnose,
        {"choose": "choose_next_strategy", "finalize_failure": "finalize_failure"},
    )

    # --- Strategy selection routing ---
    builder.add_conditional_edges(
        "choose_next_strategy",
        _route_strategy,
        {
            "traceback_sft": "attempt_sft_with_traceback",
            "local_rag": "retrieve_local_docs",
            "reflection_critic": "reflection_critic",
            "external_expert": "external_expert_repair",
            "finalize_failure": "finalize_failure",
        },
    )

    # --- RAG sub-pipeline ---
    builder.add_edge("retrieve_local_docs", "assess_local_context")
    builder.add_conditional_edges(
        "assess_local_context",
        _route_local_context,
        {"web": "web_search_docs", "summarize": "summarize_context"},
    )
    builder.add_edge("web_search_docs", "summarize_context")
    builder.add_edge("summarize_context", "attempt_sft_with_rag")

    # --- Reflection sub-pipeline ---
    builder.add_edge("reflection_critic", "attempt_sft_with_reflection")

    # --- Terminal edges ---
    builder.add_edge("finalize_success", END)
    builder.add_edge("finalize_failure", END)

    return builder.compile()

### 12) Add execution wrappers
```
# =============================================================================
# workflow execution wrappers
# =============================================================================
# Goal: Initialize clean state and provide a single run entrypoint.
# Output: Final code, status, route history, attempts, and stop reason.
```

In [ ]:
# =============================================================================
# Workflow Execution
# =============================================================================

def init_state(original_code: str, max_attempts: int = 6) -> RepairState:
    """Initialize a fresh repair state."""
    return RepairState(
        original_code=original_code,
        current_code=original_code,
        traceback="",
        check_result={},
        attempt_count=0,
        max_attempts=max_attempts,
        attempt_history=[],
        route_history=[],
        error_category="",  # Will be set by diagnose_failure
        summarized_hints="",
        reflection_feedback={},
        initial_traceback="",
        error_type="",
        error_explanation="",
        used_traceback=False,
        used_rag=False,
        used_web=False,
        used_reflection=False,
        used_external=False,
        local_docs=[],
        web_docs=[],
        local_context_quality="",
        next_strategy="",
        failure_signature="",
        previous_failure_signature="",
        repeated_failure_count=0,
        no_meaningful_change_count=0,
        should_stop=False,
        stop_reason="",
        final_status="",
        final_code="",
    )


def run_workflow(original_code: str, max_attempts: int = 6) -> dict[str, Any]:
    """
    Execute the complete repair workflow.
    
    Args:
        original_code: The broken Python code to repair
        max_attempts: Maximum repair attempts before giving up
        
    Returns:
        Dictionary with final_code, final_status, attempt_count, route_history, attempt_history
    """
    app = build_graph()
    state = init_state(original_code=original_code, max_attempts=max_attempts)
    final_state = app.invoke(state)
    
    return {
        "final_code": final_state["final_code"],
        "final_status": final_state["final_status"],
        "attempt_count": final_state["attempt_count"],
        "route_history": final_state["route_history"],
        "attempt_history": final_state["attempt_history"],
        "error_category": final_state.get("error_category", ""),
        "stop_reason": final_state.get("stop_reason", ""),
    }

### 14) Optional cleanup
```
# =============================================================================
# optional cleanup: stop llama-server
# =============================================================================
# Goal: Gracefully terminate server resources started by this notebook.
# Use: Run after experiments to release local CPU/RAM resources.
```

In [ ]:
# =============================================================================
# Cleanup: Stop llama-server (Run when done)
# =============================================================================

def stop_llama_server():
    """Stop the llama-server process if it was started by this notebook."""
    global LLAMA_SERVER_PROCESS
    if 'LLAMA_SERVER_PROCESS' in globals() and LLAMA_SERVER_PROCESS is not None:
        print("🛑 Stopping llama-server...")
        LLAMA_SERVER_PROCESS.terminate()
        LLAMA_SERVER_PROCESS.wait(timeout=5)
        LLAMA_SERVER_PROCESS = None
        print("✓ llama-server stopped")
    else:
        print("ℹ No llama-server process to stop")

# Uncomment to stop the server:
# stop_llama_server()

In [107]:
# Compact route verification for the new graph policy
from pathlib import Path

code_text = Path("test.py").read_text(encoding="utf-8")
pre = init_state(code_text, max_attempts=6)
pre = run_checks(pre)
pre_tail = [ln.strip() for ln in pre.get("traceback", "").splitlines() if ln.strip()]
print("preflight tail:", pre_tail[-1] if pre_tail else "")

res = run_workflow(code_text, max_attempts=6)
print("status:", res["final_status"])
print("error_category:", res.get("error_category", ""))
print("route:", " -> ".join(res["route_history"]))

preflight tail: 
status: success
error_category: 
route: run_checks -> finalize_success
